# OPRB test-set benchmark — Qwen2.5-VL-3B-Instruct

**Task:** Evaluate a single pretrained, lightweight vision-language model on **occluded document images**, with no fine-tuning.

| Component | Choice |
|---|---|
| Dataset | [OPRB — Occluded Pages Restoration Benchmark](https://huggingface.co/datasets/kpurkayastha/OPRB) |
| Allowed input | `test/text/img/` and `test/nontext/img/` only |
| Ground truth for scoring | Matching `test/*/gt/` token annotations (not passed to the model) |
| Model | [Qwen/Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct) (4-bit) |
| Primary metrics | Character Error Rate (CER), Word Error Rate (WER) |
| Additional metrics | Bag-of-words token precision/recall/F1 and processing time |
| Outputs | Per-image CSV, grouped CSVs, JSON configuration, result plots |

The OPRB **test set has 9,000 pages**: 6,000 `text`-region occlusions and 3,000 `nontext`-region occlusions. Test data is stored as **separate `.tar.gz` archives**, not as a directly loadable Hugging Face `Dataset` table.

**Colab instructions:** `Runtime → Change runtime type → T4 GPU` (or another CUDA GPU), then `Runtime → Run all`. Start with the small sample in Configuration. The download is still substantial because the compressed test image archives must be fetched in full. Set `MAX_SAMPLES_PER_CATEGORY = None` to evaluate all test images.

**Fair evaluation:** `img_gt/` and `annots/` are deliberately **not downloaded**; the model sees only occluded test images, never ground-truth text. No test-time parameter updates are performed. Only the held-out test split is used.

## 1. Install dependencies

In [ ]:
%pip -q install "transformers>=4.53,<5" "accelerate>=1.0" "bitsandbytes>=0.45" "huggingface_hub>=0.30" "jiwer>=3.0" "pandas>=2.0" matplotlib Pillow tqdm

## 2. Imports and GPU check

In [ ]:
import json, random, re, shutil, tarfile, time, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from huggingface_hub import hf_hub_download
from jiwer import cer, wer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "CUDA GPU required. In Colab, choose Runtime > Change runtime type > T4 GPU."
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('CUDA:', torch.version.cuda)

## 3. Configuration — edit these settings first

In [ ]:
REPO_ID = "kpurkayastha/OPRB"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DATA_ROOT = Path("/content/oprb_test")
OUTPUT_DIR = Path("/content/oprb_qwen_results")

# Both categories cover the COMPLETE test folder.
CATEGORIES = ["text", "nontext"]  # Or ["text"] to download just text-region occlusions.

# Quick smoke test; change to None for ALL 9,000 pages (if both categories are selected).
MAX_SAMPLES_PER_CATEGORY = 5
RANDOM_SEED = 42

# Tradeoff: higher image resolution and longer outputs can improve OCR but cost more GPU memory/time.
MAX_IMAGE_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 2048
NORMALIZE_CASE = True  # NFKC + whitespace normalization + casefolding

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Categories:', CATEGORIES)
print('Maximum samples per category:', MAX_SAMPLES_PER_CATEGORY)

## 4. Download and safely extract only **test images** and **test GT text**

Downloads `test/{text,nontext}/img.tar.gz` and `test/{text,nontext}/gt.tar.gz`, **not** train data, clean page images, or occluded annotations. The extraction flattens possible nested paths so that image and annotation basenames match. The first run may use several GB of disk and network bandwidth.

In [ ]:
def extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:
    """Safely copy only files of the specified type; never use archive-supplied paths."""
    destination.mkdir(parents=True, exist_ok=True)
    count = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in tqdm(archive, desc=f"Extract {destination.parent.name}/{destination.name}", leave=False):
            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):
                continue
            # Ignore any member directory traversal; write only basenames.
            filename = Path(member.name).name
            target = destination / filename
            if not target.exists():
                stream = archive.extractfile(member)
                if stream is None:
                    continue
                with stream, target.open("wb") as out:
                    shutil.copyfileobj(stream, out)
            count += 1
    return count


def download_test_category(category: str):
    if category not in {"text", "nontext"}:
        raise ValueError(f"Unknown category: {category}")
    for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):
        repo_path = f"test/{category}/{kind}.tar.gz"
        target_dir = DATA_ROOT / category / kind
        existing = list(target_dir.glob(f"*{suffix}")) if target_dir.exists() else []
        expected = 6000 if category == "text" else 3000
        if len(existing) >= expected:
            print(f"Reuse {category}/{kind}: {len(existing)} files")
            continue
        print(f"Downloading {repo_path} ...")
        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=repo_path)
        n = extract_flat(tar_path, target_dir, suffix)
        print(f"  {category}/{kind}: {n:,} archive members of type {suffix} -> {target_dir}")


for category in CATEGORIES:
    download_test_category(category)

## 5. Pair input images with evaluation annotations

In [ ]:
def parse_occlusion_metadata(stem: str):
    # OPRB filename convention: OcclusionType__density__batchID... 
    parts = stem.split("__", 2)
    return {
        "occlusion_type": parts[0] if parts else "unknown",
        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
    }


def collect_test_pairs(categories):
    pairs = []
    for category in categories:
        image_dir = DATA_ROOT / category / "img"
        gt_dir = DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())
        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")
        if not common:
            raise RuntimeError(f"No paired files for {category}. Check extraction and naming.")
        if MAX_SAMPLES_PER_CATEGORY is not None:
            rng = random.Random(RANDOM_SEED)
            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))
        for stem in common:
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                **parse_occlusion_metadata(stem),
            })
    return pairs


pairs = collect_test_pairs(CATEGORIES)
print(f"Selected {len(pairs):,} samples. Set MAX_SAMPLES_PER_CATEGORY=None to use the entire test set.")
display(pd.DataFrame(pairs)[["category", "occlusion_type", "occlusion_density", "sample_id"]].head(8))

with Image.open(pairs[0]["image_path"]) as image:
    preview = image.copy()
preview.thumbnail((650, 850))
display(preview)

## 6. Read OPRB ground-truth text and define metrics

Each GT `.txt` row has the tab-separated fields `token`, eight bbox coordinates, `font`, `category`. We extract **only field 1 (`token`)**, preserving the source row order, and join tokens into a transcript. Token ordering in a DocBank-derived annotation may differ from natural reading order on complex multi-column pages, so CER/WER should be interpreted with that limitation.

- **CER**: character-level edit distance divided by reference character count (lower is better).
- **WER**: word-level edit distance divided by reference word count (lower is better).
- **Token P/R/F1**: order-insensitive **bag-of-words** overlap, *not* bounding-box detection F1.
- Metrics can be **greater than 1.0** for very poor generations containing many insertions.

In [ ]:
def read_gt_tokens(gt_path: str):
    tokens = []
    with open(gt_path, "r", encoding="utf-8-sig", errors="replace") as file:
        for line in file:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 11 and fields[0].strip():
                tokens.append(fields[0].strip())
    return tokens


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold() if NORMALIZE_CASE else text


def bag_of_words_prf(reference: str, prediction: str):
    ref = Counter(reference.split())
    hyp = Counter(prediction.split())
    matched = sum((ref & hyp).values())
    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())
    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)
    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def compute_metrics(ground_truth: str, prediction: str):
    ref, hyp = clean_text(ground_truth), clean_text(prediction)
    if not ref:
        raise ValueError("Empty GT transcript; cannot compute meaningful CER/WER.")
    p, r, f1 = bag_of_words_prf(ref, hyp)
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }


# Check metric code on a toy example BEFORE downloading the model.
assert compute_metrics("The cat sat", "The cat")['wer'] == 1/3
print('Toy check:', compute_metrics("The cat sat", "The cat"))
print('First GT tokens:', read_gt_tokens(pairs[0]['gt_path'])[:35])

## 7. Load the VLM in 4-bit precision

**Qwen2.5-VL-3B-Instruct** is a 3-billion-parameter vision-language model capable of processing document images. The 4-bit NF4 weights help it fit on common Colab GPUs. The language model is **not trained** on OPRB data here; this is zero-shot inference.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=MAX_IMAGE_PIXELS,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
print('Model:', MODEL_ID)
print('Quantization: 4-bit NF4; no fine-tuning')
print(f'Allocated GPU memory: {torch.cuda.memory_allocated() / 2**30:.2f} GiB')

## 8. Run inference on a **single** occluded test image

In [ ]:
OCR_PROMPT = (
    "Transcribe all text visible on this document page in natural reading order. "
    "Return only the transcription, without introductory text or explanations. "
    "Do not describe the page, and do not invent text that you cannot read."
)

@torch.inference_mode()
def transcribe_page(image_path: str):
    """The model receives ONLY the occluded image and the fixed task prompt."""
    start = time.perf_counter()
    with Image.open(image_path) as im:
        image = im.convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": OCR_PROMPT},
        ],
    }]
    prompt_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[prompt_text], images=[image], padding=True, return_tensors="pt"
    ).to("cuda")
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    text = processor.decode(
        new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False
    ).strip()
    torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - start
    generated_tokens = int(new_tokens.numel())
    # Reaching max_new_tokens is a useful warning that the transcript may be cut off.
    possibly_truncated = generated_tokens >= MAX_NEW_TOKENS
    del inputs, output_ids, new_tokens
    return text, elapsed_sec, generated_tokens, possibly_truncated


sample = pairs[0]
example_prediction, example_time, example_tokens, example_truncated = transcribe_page(sample['image_path'])
example_gt = ' '.join(read_gt_tokens(sample['gt_path']))
print('CATEGORY:', sample['category'])
print('IMAGE:', Path(sample['image_path']).name)
print('LATENCY (s):', round(example_time, 2))
print('GENERATED TOKENS:', example_tokens, 'Possible truncation:', example_truncated)
print('GT (first 700 chars):', example_gt[:700])
print('PREDICTION (first 700 chars):', example_prediction[:700])
print('METRICS:', compute_metrics(example_gt, example_prediction))

## 9. Evaluate selected test pages (no training)

This is **inference only**. Long document pages may exceed the output token budget; a `possibly_truncated` column is provided, and such results need to be considered in the report. Failures are logged separately rather than silently scored as blank predictions. Each successful prediction is saved to a checkpoint CSV during the loop.

In [ ]:
RESULTS_CSV = OUTPUT_DIR / 'per_image_results.csv'

# Resume safely when a run is interrupted (for same settings/model/prompt).
if RESULTS_CSV.exists():
    previous = pd.read_csv(RESULTS_CSV).to_dict(orient='records')
    rows = [r for r in previous if r.get('sample_id') in {p['sample_id'] for p in pairs}]
    print(f'Resuming with {len(rows)} previously saved rows.')
else:
    rows = []

finished = {r['sample_id'] for r in rows}
for sample in tqdm(pairs, desc='OPRB test evaluation'):
    if sample['sample_id'] in finished:
        continue
    result = dict(sample)
    result['model_id'] = MODEL_ID
    try:
        prediction, elapsed, n_tokens, truncated = transcribe_page(sample['image_path'])
        ground_truth = ' '.join(read_gt_tokens(sample['gt_path']))
        result.update({
            'status': 'ok',
            'prediction': prediction,
            'ground_truth': ground_truth,
            'latency_sec': elapsed,
            'generated_tokens': n_tokens,
            'possibly_truncated': truncated,
            **compute_metrics(ground_truth, prediction),
        })
    except Exception as exc:
        result.update({'status': 'error', 'error': repr(exc)})
        print('Failure:', sample['sample_id'], repr(exc))
    rows.append(result)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
print('Saved:', RESULTS_CSV)
print('Successful:', (results.status == 'ok').sum(), '/', len(results))
display(results[['category', 'occlusion_type', 'cer', 'wer', 'token_f1', 'latency_sec', 'possibly_truncated', 'status']].head(10))

## 10. Summarize overall results and compare occlusion categories/types

In [ ]:
ok = results.loc[results['status'] == 'ok'].copy()
assert len(ok), 'No successful predictions. Inspect the error column.'

metric_cols = ['cer', 'wer', 'token_precision', 'token_recall', 'token_f1', 'latency_sec']

def aggregate_metrics(df, group_by):
    summary = df.groupby(group_by, dropna=False).agg(
        n=('sample_id', 'count'),
        **{f'mean_{c}': (c, 'mean') for c in metric_cols},
        truncated=('possibly_truncated', 'sum'),
    ).reset_index()
    return summary

# IMPORTANT: These are macro averages across pages, NOT corpus WER/CER.
overall = {f'mean_{m}': float(ok[m].mean()) for m in metric_cols}
overall.update({
    'n_success': int(len(ok)),
    'n_errors': int(len(results) - len(ok)),
    'n_possibly_truncated': int(ok['possibly_truncated'].sum()),
})

by_category = aggregate_metrics(ok, ['category'])
by_type = aggregate_metrics(ok, ['category', 'occlusion_type'])
by_density = aggregate_metrics(ok, ['category', 'occlusion_density'])

print('OVERALL macro-averaged metrics:')
print(json.dumps(overall, indent=2))
print('\nBY OCCLUSION REGION:')
display(by_category)
print('\nBY OCCLUSION TYPE:')
display(by_type)
print('\nBY OCCLUSION DENSITY:')
display(by_density)

by_category.to_csv(OUTPUT_DIR / 'summary_by_category.csv', index=False)
by_type.to_csv(OUTPUT_DIR / 'summary_by_occlusion_type.csv', index=False)
by_density.to_csv(OUTPUT_DIR / 'summary_by_density.csv', index=False)

## 11. Visualize the errors

In [ ]:
ax = by_category.set_index('category')[['mean_cer', 'mean_wer']].plot(
    kind='bar', figsize=(7, 4), rot=0, title='OPRB test: mean CER and WER by region'
)
ax.set_xlabel('Occlusion region')
ax.set_ylabel('Error rate (lower is better)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cer_wer_by_category.png', dpi=150)
plt.show()

ax = by_type.set_index('occlusion_type').groupby(level=0)['mean_wer'].mean().sort_values().plot(
    kind='barh', figsize=(9, 5), title='Mean WER by occlusion type (unweighted category average)'
)
ax.set_xlabel('Mean word error rate (lower is better)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'wer_by_occlusion_type.png', dpi=150)
plt.show()

## 12. Export results (.zip) and reproducibility settings

In [ ]:
from importlib.metadata import version

run_config = {
    'dataset': REPO_ID,
    'split': 'test',
    'categories': CATEGORIES,
    'model': MODEL_ID,
    'precision': 'NF4 4-bit weights; float16 computation',
    'no_finetuning': True,
    'max_samples_per_category': MAX_SAMPLES_PER_CATEGORY,
    'random_seed': RANDOM_SEED,
    'max_image_pixels': MAX_IMAGE_PIXELS,
    'max_new_tokens': MAX_NEW_TOKENS,
    'normalize_case': NORMALIZE_CASE,
    'prompt': OCR_PROMPT,
    'metrics': overall,
    'metric_definition': 'Macro means of per-image CER/WER and bag-of-words overlap',
    'transformers_version': version('transformers'),
    'torch_version': torch.__version__,
    'gpu_name': torch.cuda.get_device_name(0),
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_name = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Results ZIP:', zip_name)

# Download from Colab. Outside Colab this step is optional.
try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    print('Not running in Google Colab; results remain at', OUTPUT_DIR)

## Student reporting checklist and limitations

Submit your `oprb_qwen_results.zip` plus a short report that includes the **model name, GPU, resolution, output-token limit, number of evaluated pages, mean CER/WER, token F1, average latency**, and separate results for the `text` and `nontext` subsets. Include at least three qualitative examples of successes and failures from `per_image_results.csv`.

**Important interpretation notes**

1. This notebook evaluates **page-level transcription against the clean-page annotations**, not complete image restoration, font recovery, geometry reconstruction, or localization.
2. The model is instructed **not to invent hidden text**. Occluded tokens in the clean-page GT may therefore penalize faithful transcription even when the model correctly acknowledges that text is unreadable. The scores are best viewed as *full-page text recovery* under occlusion, rather than visible-only OCR accuracy.
3. Matching GT tokens are joined in annotation row order. This is an approximation of page reading order; multi-column pages, equations and special characters can increase measured CER/WER without equivalent recognition failures.
4. The 4-bit model and image downscaling introduce compute/accuracy tradeoffs. A large `max_new_tokens` limit reduces output truncation but does not guarantee complete pages.
5. Scores from 5 pages per category are **only a smoke test**, not a representative benchmark. For the official evaluation, set `MAX_SAMPLES_PER_CATEGORY = None` and clear the checkpoint CSV if you have changed the model, prompt, or metrics.
6. The results directory stores the ground-truth text in CSV for transparent scoring. **Never pass those columns back to the model as input.**

**Sources:** [OPRB dataset card](https://huggingface.co/datasets/kpurkayastha/OPRB) · [Qwen2.5-VL-3B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct).